In [0]:
# Same pinned set as s05_oot / s09 (the project has no requirements.lock yet)
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
# s06b: Cut-off analysis for the locked FTS model v1.0.1, Models A and C (spec 8.2, 8.3, 9.1-9.3; E13 scoring code)
# REPORT ONLY. APPROVED APPLICANTS ONLY (acard_master). Writes only under <artifacts_root>/<config.cutoff_analysis.output_folder>/
# (plus the owner-requested entry in reports/open_questions.md).
# Owner decisions 2026-11-17: approval-rate base = config.psi_statuses (GOOD, BAD, INDETERMINATE, IMMATURE, EXCLUDED_PERF);
# bad rates from GOOD + BAD only; INDETERMINATE / IMMATURE / EXCLUDED_PERF reported as separate counts; EXCLUDED not used;
# segments: WATCH variables only (config.columns.segments are not in acard_master -> open question).
import sys, os, json, time, pickle, shutil, tempfile, hashlib, importlib, warnings, re
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr, chi2_contingency
from pyspark.sql import functions as F

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="Missing values detected")

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)
CA = cfg["cutoff_analysis"]
MODELS_USED = list(CA["models"])
assert MODELS_USED == ["A", "C"], f"this module scores Models A and C only, config has {MODELS_USED}"
assert CA["performance_window"] == "OOT", "config.cutoff_analysis.performance_window must be OOT"
GRID = [float(r) for r in CA["approval_rate_grid"]]
BW = int(CA["score_band_width"])
COMPARE_WITH = CA.get("compare_with")
# Benchmark (spec 8.3): config.cutoff_analysis.compare_with = {name, table, key, score_column, date_column, higher_is_better}
BENCH, BENCH_NAME = None, None
if COMPARE_WITH:
    _need = {"name", "table", "key", "score_column", "higher_is_better"}
    assert isinstance(COMPARE_WITH, dict) and _need <= set(COMPARE_WITH), f"compare_with needs {sorted(_need)}"
    assert COMPARE_WITH["higher_is_better"] is True, "only higher-is-better benchmark scores are supported (approve if score >= cut-off)"
    BENCH, BENCH_NAME = "ASCORE", str(COMPARE_WITH["name"])

ARTIFACTS = common.artifacts_path(cfg)
REG_PATH = os.path.join(ARTIFACTS, "model_registry.json")
OUT = os.path.join(ARTIFACTS, CA["output_folder"])
os.makedirs(os.path.join(OUT, "run_summaries"), exist_ok=True)
VERSION = str(cfg["registry"]["version"])
XLSX_PATH = os.path.join(OUT, f"cutoff_analysis_v{VERSION}_A_C.xlsx")
SUMMARY_PATH = os.path.join(OUT, "run_summaries", "s06b_cutoff_analysis.json")
OQ_PATH = os.path.join(ARTIFACTS, "reports", "open_questions.md")    # owner request: the only write outside OUT

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.{cfg['tables']['master']}"
E13_STORED = f"{cat}.{sch}.acard_oot_predictions"     # E13 case-level OOT scores, used only for the spot check
id_col, DATE_A, DECISION_A = cfg["columns"]["id"], cfg["columns"]["date"], cfg["columns"]["decision_time"]
LABELLED = list(cfg["labelled_statuses"])
BASE_STATUSES = list(cfg["psi_statuses"])
assert set(LABELLED) == {"GOOD", "BAD"}
assert set(BASE_STATUSES) == {"GOOD", "BAD", "INDETERMINATE", "IMMATURE", "EXCLUDED_PERF"}, f"unexpected psi_statuses {BASE_STATUSES}"
OTHER = ["INDETERMINATE", "IMMATURE", "EXCLUDED_PERF"]       # in the approval base, never in bad rates
BOOT = cfg["selection"]["bootstrap"]
SPLITS = cfg["oot_report_splits"]
WATCH = list(cfg["column_roles"]["watch"])
SEGMENTS = cfg["columns"].get("segments") or []
EPS = cfg["funnel"]["L3"]["eps"]

CAVEAT = ("Approval rates are relative to previously approved applicants, not to all applicants; "
          "they overstate the through-the-door approval rate.")

# Report-only constants from the request (nothing is selected, dropped or fitted with them)
SEG_MIN_BADS = 50                      # STEP 4.3: segments with < 50 bads -> bad rate and count only (no CI)
AGE_BANDS = [0, 25, 30, 35, 40, 45, 50, 60, 200]   # same bands as s09
TOP_N_WATCH = 10                       # other categories -> OTHER
IMM_RATIO_FLAG = 1.5                   # IMMATURE flag: rate ratio vs the comparison group
IMM_BAND_MIN_N = 30                    # IMMATURE flag: minimum band size
IMM_P_FLAG = 0.05                      # IMMATURE flag: chi-square p-value

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
RUN_TIME = datetime.now().isoformat(timespec="seconds")
_norm = common._norm
WRITTEN = []

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, float) and np.isnan(o):
        return None
    return str(o)

def _guard(dst):
    ok = os.path.abspath(dst).startswith(os.path.abspath(OUT) + os.sep) or os.path.abspath(dst) == os.path.abspath(OQ_PATH)
    assert ok, f"refusing to write outside {OUT}: {dst}"

def write_text(text, dst):
    """Local temp file, then copy to the UC volume (copyfile: FUSE does not allow chmod/utime)."""
    _guard(dst)
    tmp = os.path.join(tempfile.mkdtemp(prefix="s06b_"), os.path.basename(dst))
    with open(tmp, "w") as fh:
        fh.write(text)
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

def write_json(obj, dst):
    write_text(json.dumps(obj, indent=2, default=_json_default), dst)

def save_excel(sheets, dst):
    """sheets: name -> DataFrame, or list of (title, DataFrame) blocks. Row 1 of EVERY sheet is the approved-only caveat."""
    _guard(dst)
    tmp = os.path.join(tempfile.mkdtemp(prefix="s06b_"), os.path.basename(dst))
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        for name, obj in sheets.items():
            assert len(name) <= 31, f"sheet name too long: {name}"
            blocks = obj if isinstance(obj, list) else [(None, obj)]
            pd.DataFrame(columns=[CAVEAT]).to_excel(xw, sheet_name=name, startrow=0, index=False)
            r = 2
            for title, df in blocks:
                df = df.copy()
                for c in df.columns:
                    if isinstance(df[c].dtype, pd.PeriodDtype) or str(df[c].dtype) in ("string", "category") or str(df[c].dtype).startswith("interval"):
                        df[c] = df[c].astype(str)
                if title:
                    pd.DataFrame(columns=[title]).to_excel(xw, sheet_name=name, startrow=r, index=False)
                    r += 1
                df.to_excel(xw, sheet_name=name, startrow=r, index=False)
                r += len(df) + 3
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}\nOutput      : {OUT}\nWorkbook    : {XLSX_PATH}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | grid {GRID} | band width {BW} | compare_with {COMPARE_WITH}")

In [0]:
assert os.path.exists(REG_PATH), f"ABORT: registry not found: {REG_PATH}"
REG = read_json(REG_PATH)
assert REG.get("locked") is True, "ABORT: registry is not locked"
for k in ("decision_rules_signed_by", "signed_date"):
    assert str(REG.get("signoff", {}).get(k) or "").strip(), f"ABORT: sign-off field {k} is empty"
assert str(REG["version"]) == VERSION, f"ABORT: registry version {REG['version']} != config.registry.version {VERSION}"
REG_SHA = sha256_file(REG_PATH)
FILES = {k: v["path"] for k, v in REG["files"].items()}
LOCKED_SHA = {k: v["sha256"] for k, v in REG["files"].items()}
bad = [k for k, p in FILES.items() if not os.path.exists(p) or sha256_file(p) != LOCKED_SHA[k]]
assert not bad, f"ABORT: {len(bad)} locked model/binning file(s) changed since the lock: {bad}"
need = ["models/A/binning_process.pkl", "models/A/lr_sklearn.pkl", "models/C/lgbm.txt", "models/model_spec.json", "oof_predictions.parquet"]
assert not [k for k in need if k not in FILES], f"ABORT: registry lacks {[k for k in need if k not in FILES]}"

# Platt calibration (fitted on DEV OOF in E13): hash recorded in registry.oot_evaluation.calibration_file
CAL_REF = REG["oot_evaluation"]["calibration_file"]
PLATT_PATH = CAL_REF["path"]
PLATT_SHA = sha256_file(PLATT_PATH)
assert PLATT_SHA == CAL_REF["sha256"], "ABORT: Platt calibration file differs from the hash in the registry"
PLATT_DOC = read_json(PLATT_PATH)
assert str(PLATT_DOC["registry_version"]) == VERSION and PLATT_DOC["method"] == "platt" and PLATT_DOC["fit_on"] == "dev_oof", "ABORT: Platt file is not the DEV-OOF Platt fit of this version"
assert PLATT_DOC["oof_sha256"] == LOCKED_SHA["oof_predictions.parquet"], "ABORT: Platt fit on a different OOF file"
PLATT = {m: PLATT_DOC["params"][m] for m in MODELS_USED}

SPEC = read_json(FILES["models/model_spec.json"])
A_FEATS, POOL, BC_COLS = REG["features"]["A"], REG["features"]["BC_pool"], REG["features"]["BC_columns"]
FEATS = list(dict.fromkeys(A_FEATS + POOL))
WOE_KW = SPEC["A"]["woe_transform"]
A_BP = pickle.load(open(FILES["models/A/binning_process.pkl"], "rb"))
A_LR = pickle.load(open(FILES["models/A/lr_sklearn.pkl"], "rb"))
C_BST = lgb.Booster(model_file=FILES["models/C/lgbm.txt"])
assert C_BST.num_trees() == REG["lightgbm_rounds"]["final_rounds"], "ABORT: LightGBM trees != locked final rounds"
FDICT_PATH = os.path.join(PROJECT_ROOT, "config", "feature_dict.csv")
FDICT, FDICT_SHA = pd.read_csv(FDICT_PATH), sha256_file(FDICT_PATH)
MODEL_ROOT = os.path.dirname(REG_PATH)

def tree_snapshot(root):
    """(size, mtime) of every file under the model folder, excluding the output folder and the open-questions file."""
    snap, out_abs, oq_abs = {}, os.path.abspath(OUT), os.path.abspath(OQ_PATH)
    for dp, _, fn in os.walk(root):
        a = os.path.abspath(dp)
        if a == out_abs or a.startswith(out_abs + os.sep):
            continue
        for f in fn:
            p = os.path.join(dp, f)
            if os.path.abspath(p) == oq_abs:
                continue
            st = os.stat(p)
            snap[p] = (st.st_size, st.st_mtime)
    return snap

def table_version(t):
    return int(spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").first()["version"])

PROTECTED_TABLES = {}
for t in [MASTER_TBL, f"{cat}.{sch}.acard_feature_log", f"{cat}.{sch}.{cfg['tables']['feature_log']}", E13_STORED]:
    if spark.catalog.tableExists(t):
        PROTECTED_TABLES[t] = table_version(t)
SNAP0 = tree_snapshot(MODEL_ROOT)
print(f"v{REG['version']} locked {REG['locked_at']} | signed {REG['signoff']['decision_rules_signed_by']} {REG['signoff']['signed_date']} | "
      f"{len(FILES)} locked hashes OK | Platt OK ({PLATT_SHA[:12]})")
for m in MODELS_USED:
    print(f"  Platt {m}: a={PLATT[m]['a']:.4f} b={PLATT[m]['b']:.4f} (n={PLATT[m]['n']}, bads={PLATT[m]['n_bad']})")
print(f"A: {len(A_FEATS)} features | C: {len(POOL)} pool features, {C_BST.num_trees()} trees")
print(f"Snapshot: {len(SNAP0)} files under {MODEL_ROOT} | protected tables {PROTECTED_TABLES}")

In [0]:
m_cols = spark.table(MASTER_TBL).columns
WATCH_COLS = [w for w in WATCH if w in m_cols]
SEG_AVAILABLE = [c for c in SEGMENTS if c in m_cols]
SEG_MISSING = [c for c in SEGMENTS if c not in m_cols]
assert not [f for f in FEATS if f not in m_cols], "model features missing in master"
keep = list(dict.fromkeys([id_col, "CREDITCARDNO", DATE_A, DECISION_A, "window", "sample_status"] + WATCH_COLS + SEG_AVAILABLE + FEATS))
master = spark.table(MASTER_TBL).select(*keep).toPandas()
STATUS_COUNTS = master.groupby(["window", "sample_status"]).size().unstack(fill_value=0).reset_index()

def scope(name):
    """One window via common.load_window (OOT read lock), approval base = config.psi_statuses (EXCLUDED left out)."""
    df = common.load_window(master, name, cfg, registry_path=REG_PATH)
    df = df[df["sample_status"].isin(BASE_STATUSES)].reset_index(drop=True)
    d, a = pd.to_datetime(df[DATE_A]), pd.to_datetime(df[DECISION_A])
    df["app_month"] = d.dt.to_period("M").astype(str)
    df["approve_month"] = a.dt.to_period("M").astype(str)
    df["gap_days"] = (a - d).dt.days
    return df

OOT, FWD, STRESS = scope("OOT"), scope("FWD"), scope("STRESS")
_d = pd.to_datetime(OOT[DATE_A])
OOT["oot_split"] = None
for q, (s, e) in SPLITS.items():
    OOT.loc[(_d >= s) & (_d < e), "oot_split"] = q
assert OOT["oot_split"].notna().all(), "OOT rows outside config.oot_report_splits"

def score_e13(df):
    """E13 scoring (s05_oot 'Score with the locked models'): raw PD -> Platt (DEV OOF) -> common score scaling."""
    XA = df[A_FEATS].apply(pd.to_numeric, errors="coerce")
    raw = {"A": A_LR.predict_proba(np.asarray(A_BP.transform(XA, metric="woe", **WOE_KW), dtype=float))[:, 1]}
    XR = common.prepare_raw(df, POOL, FDICT)
    assert list(XR.columns) == BC_COLS, "prepare_raw columns differ from the locked B/C columns"
    raw["C"] = C_BST.predict(XR)
    for m in MODELS_USED:
        cal = common.apply_platt(raw[m], PLATT[m])
        assert np.isfinite(cal).all() and ((cal > 0) & (cal < 1)).all(), f"{m}: PD outside (0, 1)"
        df[f"raw_{m}"], df[f"pd_{m}"], df[f"score_{m}"] = raw[m], cal, common.pd_to_score(cal, cfg)
    return df

SCORED_WINDOWS = []
for nm, df in (("OOT", OOT), ("FWD", FWD), ("STRESS", STRESS)):
    score_e13(df)
    SCORED_WINDOWS.append(nm)
assert "DEV" not in SCORED_WINDOWS                      # never score DEV in-sample

# Spot check: OOT scores reproduce the stored E13 scores (1,000 rows)
st = spark.table(E13_STORED).select("CREDITCARDNO", *[f"prob_{m}_cal" for m in MODELS_USED], *[f"score_{m}" for m in MODELS_USED]).toPandas()
st["CREDITCARDNO"] = st["CREDITCARDNO"].astype(str)
st = st[~st["CREDITCARDNO"].duplicated(keep=False)].set_index("CREDITCARDNO")
card = OOT["CREDITCARDNO"].astype(str)
cand = sorted(set(card[~card.duplicated(keep=False)]) & set(st.index))
spot = set(np.random.default_rng(SEED).choice(cand, size=min(1000, len(cand)), replace=False))
sel = np.where(card.isin(spot).to_numpy())[0]
SPOT = []
for m in MODELS_USED:
    s_ = st.loc[card.iloc[sel]]
    SPOT.append({"model": m, "stored_table": E13_STORED, "n": len(sel),
                 "max_abs_pd_diff": float(np.abs(OOT[f"pd_{m}"].to_numpy()[sel] - s_[f"prob_{m}_cal"].to_numpy()).max()),
                 "n_score_diff": int((OOT[f"score_{m}"].to_numpy()[sel] != s_[f"score_{m}"].to_numpy()).sum())})
SPOT = pd.DataFrame(SPOT)

# DEV second view: each model's OOF predictions (never in-sample DEV scores), Platt-calibrated and scaled
OOF_RAW = pd.read_parquet(FILES["oof_predictions.parquet"])
OOF_RAW["app_id"] = OOF_RAW["app_id"].astype(str)
dev = common.load_window(master, "DEV", cfg, registry_path=REG_PATH)
dev_gb = dev[dev["sample_status"].isin(LABELLED)][[id_col, "sample_status", DATE_A] + WATCH_COLS].copy()
dev_gb[id_col] = dev_gb[id_col].astype(str)
DEVO = {}
for m in MODELS_USED:
    o = OOF_RAW[OOF_RAW["model"] == m][["app_id", "fold", "p_oof"]]
    assert o["app_id"].is_unique and o["p_oof"].notna().all(), f"OOF {m}: duplicate ids or missing predictions"
    j = o.merge(dev_gb, left_on="app_id", right_on=id_col, how="left", validate="1:1")
    assert j["sample_status"].notna().all(), f"OOF {m}: rows that are not DEV GOOD/BAD in master"
    j[f"pd_{m}"] = common.apply_platt(j["p_oof"].to_numpy(), PLATT[m])
    j[f"score_{m}"] = common.pd_to_score(j[f"pd_{m}"].to_numpy(), cfg)
    j["pd_source"] = "oof_predictions.parquet (p_oof) + Platt"
    DEVO[m] = j.reset_index(drop=True)

OOT_COUNTS = OOT["sample_status"].value_counts().reindex(BASE_STATUSES, fill_value=0)
for df in (OOT, FWD, STRESS):
    df["_row"] = np.arange(len(df))            # row order guard for the benchmark merge
print("OOT base:", len(OOT), OOT_COUNTS.to_dict(), "| by split:", OOT["oot_split"].value_counts().to_dict())
print("FWD base:", len(FWD), FWD["sample_status"].value_counts().to_dict(), "| STRESS base:", len(STRESS), STRESS["sample_status"].value_counts().to_dict())
print("DEV OOF:", {m: (len(d), int((d['sample_status'] == 'BAD').sum())) for m, d in DEVO.items()}, "| DEV GOOD+BAD in master:", len(dev_gb))
print("Segments available:", SEG_AVAILABLE, "| missing:", SEG_MISSING, "| watch:", WATCH_COLS)
display(SPOT)

In [0]:
def load_a_score(df_target, table, key="CRNNO"):
    """Owner's A-Score loader (2026-11-17): A-Score rows for the CRNNOs in df_target, merged on CRNNO (right join).
    Table and key come from config.cutoff_analysis.compare_with; df_target is not modified."""
    a_score = spark.table(table)
    crnno_list = [str(c) for c in df_target[key].unique().tolist()]
    a_score_filtered = (a_score.withColumn(key, F.col(key).cast("string")).filter(F.col(key).isin(crnno_list)))
    a_score_pd = a_score_filtered.toPandas()
    a_score_pd[key] = a_score_pd[key].astype(str)
    assert a_score_pd[key].is_unique, f"{table}: more than one row per {key}"
    df_target = df_target.copy()
    df_target[key] = df_target[key].astype(str)
    a_score_pd = a_score_pd.rename(columns={c: f"bench_{c}" for c in a_score_pd.columns if c != key})
    a_score_merged = a_score_pd.merge(df_target, on=key, how="right", validate="1:m")
    print(f"A-Score raw shape      : {a_score_pd.shape}")
    print(f"A-Score merged shape   : {a_score_merged.shape}")
    print(f"Unique CRNNOs matched  : {a_score_pd[key].nunique():,} of {df_target[key].nunique():,}")
    print(f"Columns: {list(a_score_pd.columns)}")
    return a_score_merged

BENCH_COVERAGE, BENCH_TIMING, BENCH_OFFSETS = None, None, None
if BENCH:
    CW = COMPARE_WITH
    assert CW["key"] == id_col, f"benchmark key {CW['key']} must be the application id {id_col}"
    assert CW.get("date_column"), "compare_with.date_column is required for the score-month rule"
    # Owner decision 2026-11-17: use an A-Score only if its score month (date_column) is the application month or the
    # month after (offsets 0 and 1); other rows count as 'no A-Score'. Overridable by compare_with.allowed_score_month_offsets.
    BENCH_OFFSETS = sorted(int(v) for v in CW.get("allowed_score_month_offsets", [0, 1]))
    cov, timing = [], []
    for w, df in (("OOT", OOT), ("FWD", FWD), ("STRESS", STRESS)):
        mg = load_a_score(df[[id_col, "_row"]], CW["table"], key=id_col).sort_values("_row").reset_index(drop=True)
        assert (mg["_row"].to_numpy() == df["_row"].to_numpy()).all(), f"{w}: benchmark merge changed rows"
        sc = pd.to_numeric(mg[f"bench_{CW['score_column']}"], errors="coerce")
        bd = pd.to_datetime(mg[f"bench_{CW['date_column']}"])
        gap = (bd.dt.to_period("M") - pd.to_datetime(df[DATE_A]).dt.to_period("M")).map(lambda x: x.n if pd.notna(x) else np.nan)
        any_sc = sc.notna().to_numpy()
        in_win = any_sc & gap.isin(BENCH_OFFSETS).to_numpy()
        df[f"bench_month_offset_{BENCH}"] = gap.to_numpy()
        df[f"score_{BENCH}"] = np.where(in_win, sc.to_numpy(), np.nan)
        df[f"pd_{BENCH}"], df[f"has_{BENCH}"] = np.nan, in_win
        st = df["sample_status"]
        cov.append({"window": w, "n": len(df), "n_with_any_score": int(any_sc.sum()),
                    "n_score_outside_allowed_months": int((any_sc & ~in_win).sum()),
                    "n_with_score_used": int(in_win.sum()), "coverage_used": in_win.mean(),
                    **{f"coverage_used_{s}": in_win[(st == s).to_numpy()].mean() if (st == s).any() else np.nan for s in BASE_STATUSES}})
        vc = gap[any_sc].value_counts().sort_index()
        timing += [{"window": w, "months_score_date_minus_application": int(k), "n": int(v), "used": int(k) in BENCH_OFFSETS}
                   for k, v in vc.items()]
    BENCH_COVERAGE, BENCH_TIMING = pd.DataFrame(cov), pd.DataFrame(timing)
    print(f"A-Score used only for score month - application month in {BENCH_OFFSETS}")
    display(BENCH_COVERAGE)
    if len(BENCH_TIMING):
        display(BENCH_TIMING[BENCH_TIMING["window"] == "OOT"])
else:
    print("compare_with is null: no benchmark comparison")

In [0]:
def _counts(st, mask=None):
    st = st if mask is None else st[mask]
    return {s: int((st == s).sum()) for s in ["GOOD", "BAD"] + OTHER}

def boot_idx(bad_gb, key):
    """Stratified (GOOD/BAD) bootstrap indices over the GOOD+BAD rows; seed = config.selection.bootstrap.seed + fixed key."""
    rng = np.random.default_rng([BOOT["seed"], key])
    ib, ig = np.where(bad_gb)[0], np.where(~bad_gb)[0]
    if not BOOT.get("stratified", True):
        return rng.choice(len(bad_gb), (BOOT["n"], len(bad_gb))).astype(np.int32)
    return np.concatenate([rng.choice(ib, (BOOT["n"], len(ib))), rng.choice(ig, (BOOT["n"], len(ig)))], axis=1).astype(np.int32)

def band_lower(score):
    return (np.floor(np.asarray(score, dtype=float) / BW) * BW).astype(int)

def band_table(F, m):
    """Score bands of BW points, best (highest score) first; approved if score >= band lower bound."""
    sc, pd_, st = F[f"score_{m}"].to_numpy(), F[f"pd_{m}"].to_numpy(), F["sample_status"].to_numpy()
    lb, gb, bad = band_lower(sc), np.isin(st, LABELLED), st == "BAD"
    N, TB, TGB = len(F), int(bad.sum()), int(gb.sum())
    rows = []
    for b in sorted(np.unique(lb), reverse=True):
        mk = lb == b
        c = _counts(st, mk)
        ngb = c["GOOD"] + c["BAD"]
        rows.append({"band": f"{b}-{b + BW - 1}", "band_lower": b, "n": int(mk.sum()), "share": mk.sum() / N, **c,
                     "n_good_bad": ngb, "bad_rate": c["BAD"] / ngb if ngb else np.nan,
                     "good_bad_odds": c["GOOD"] / c["BAD"] if c["BAD"] else np.nan,
                     "mean_pd_good_bad": pd_[mk & gb].mean() if (mk & gb).any() else np.nan})
    t = pd.DataFrame(rows)
    t["pd_minus_bad_rate"] = t["mean_pd_good_bad"] - t["bad_rate"]
    t["cum_n_approved"] = t["n"].cumsum()
    t["cum_approval_rate"] = t["cum_n_approved"] / N
    t["cum_good_bad_approved"] = t["n_good_bad"].cumsum()
    t["cum_approval_rate_good_bad_only"] = t["cum_good_bad_approved"] / TGB
    t["cum_bads_approved"] = t["BAD"].cumsum()
    t["cum_approved_bad_rate"] = t["cum_bads_approved"] / t["cum_good_bad_approved"].replace(0, np.nan)
    t["cum_bads_declined"] = TB - t["cum_bads_approved"]
    t["cum_share_of_bads_declined"] = t["cum_bads_declined"] / TB
    for s in OTHER:
        t[f"cum_{s}_approved"] = t[s].cumsum()
    prev = t["bad_rate"].ffill().shift(1)            # bad rate of the nearest better band with GOOD/BAD rows
    t["non_monotonic"] = t["bad_rate"].notna() & prev.notna() & (t["bad_rate"] < prev)
    return t

BANDS = {m: band_table(OOT, m) for m in MODELS_USED}
for m, t in BANDS.items():
    assert t["n"].sum() == len(OOT) and t["BAD"].sum() == OOT_COUNTS["BAD"] and t["GOOD"].sum() == OOT_COUNTS["GOOD"]
    print(f"{m}: {len(t)} bands {t['band_lower'].min()}..{t['band_lower'].max()} | non-monotonic bands: {int(t['non_monotonic'].sum())} "
          f"| bands with < 30 GOOD+BAD: {int((t['n_good_bad'] < 30).sum())}")
display(BANDS["A"][["band", "n", "share", "GOOD", "BAD", "INDETERMINATE", "IMMATURE", "EXCLUDED_PERF", "bad_rate", "mean_pd_good_bad", "cum_approval_rate", "cum_approved_bad_rate", "non_monotonic"]])

In [0]:
def cutoff_for_rate(score, r):
    """Highest integer cut-off with approval rate (score >= cut-off) >= r."""
    s = np.sort(np.asarray(score))[::-1]
    return int(s[max(int(np.ceil(r * len(s))) - 1, 0)])

def grid_table(F, m, I, cuts=None, others_available=True, view="OOT"):
    """One row per grid approval rate. cuts: {rate: cut-off} to apply fixed cut-offs (else derived on F).
    I: bootstrap indices over F's GOOD+BAD rows (in row order)."""
    sc, pd_, st = F[f"score_{m}"].to_numpy(), F[f"pd_{m}"].to_numpy(), F["sample_status"].to_numpy()
    gb, bad = np.isin(st, LABELLED), st == "BAD"
    bad_gb = bad[gb]
    N, TB, TGB = len(F), int(bad.sum()), int(gb.sum())
    rows = []
    for r in GRID:
        c = cuts[r] if cuts is not None else cutoff_for_rate(sc, r)
        a = sc >= c
        a_gb = a[gb]
        n_app, gb_app, b_app = int(a.sum()), int(a_gb.sum()), int((a & bad).sum())
        row = {"view": view, "model": m, "approval_rate_target": r, "cutoff_score": c,
               "n_tied_at_cutoff": int((sc == c).sum()), "approval_rate_achieved": n_app / N,
               "approval_rate_if_cutoff_plus_1": float((sc > c).mean()),
               "approval_rate_good_bad_only": gb_app / TGB,
               "n_total": N, "n_approved": n_app, "n_declined": N - n_app,
               "good_bad_total": TGB, "good_bad_approved": gb_app, "good_bad_declined": TGB - gb_app,
               "bads_total": TB, "bads_approved": b_app, "bads_declined": TB - b_app,
               "share_of_bads_declined": (TB - b_app) / TB,
               "approved_bad_rate": b_app / gb_app if gb_app else np.nan}
        bb = (bad_gb & a_gb)[I].sum(axis=1)
        nb = a_gb[I].sum(axis=1)
        br = bb / np.where(nb == 0, np.nan, nb)
        row["approved_bad_rate_ci_low"], row["approved_bad_rate_ci_high"] = np.nanpercentile(br, 2.5), np.nanpercentile(br, 97.5)
        row["declined_bad_rate"] = (TB - b_app) / (TGB - gb_app) if TGB - gb_app else np.nan
        row["expected_bad_rate_approved"] = pd_[a & gb].mean() if (a & gb).any() else np.nan
        row["expected_minus_actual"] = row["expected_bad_rate_approved"] - row["approved_bad_rate"]
        for s in OTHER:
            if others_available:
                ns = st == s
                row[f"{s}_approved"], row[f"{s}_declined"] = int((ns & a).sum()), int((ns & ~a).sum())
                row[f"{s}_share_among_approved"] = (ns & a).sum() / n_app if n_app else np.nan
            else:
                row[f"{s}_approved"] = row[f"{s}_declined"] = row[f"{s}_share_among_approved"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

GB_OOT = OOT["sample_status"].isin(LABELLED).to_numpy()
I_OOT = boot_idx((OOT["sample_status"] == "BAD").to_numpy()[GB_OOT], 1)
GRID_OOT = {m: grid_table(OOT, m, I_OOT, view="OOT") for m in MODELS_USED}
CUTS = {m: dict(zip(GRID_OOT[m]["approval_rate_target"], GRID_OOT[m]["cutoff_score"])) for m in MODELS_USED}

# DEV second view: OOF predictions exist only for DEV GOOD + BAD, so the base there is GOOD + BAD
# (INDETERMINATE / IMMATURE / EXCLUDED_PERF have no OOF prediction and in-sample scores are not allowed).
GRID_DEV = {}
for k, m in enumerate(MODELS_USED):
    F = DEVO[m]
    I_dev = boot_idx((F["sample_status"] == "BAD").to_numpy(), 10 + k)
    GRID_DEV[m] = grid_table(F, m, I_dev, others_available=False, view="DEV_OOF (base = GOOD+BAD)")

for m in MODELS_USED:
    print(m, "OOT cut-offs:", CUTS[m])
cols = ["model", "approval_rate_target", "cutoff_score", "n_tied_at_cutoff", "approval_rate_achieved", "approval_rate_good_bad_only",
        "approved_bad_rate", "approved_bad_rate_ci_low", "approved_bad_rate_ci_high", "declined_bad_rate", "share_of_bads_declined",
        "expected_bad_rate_approved", "INDETERMINATE_share_among_approved", "IMMATURE_share_among_approved"]
display(pd.concat([GRID_OOT[m][cols] for m in MODELS_USED]))

In [0]:
def compare_models(F, x, y, I, cuts_x, cuts_y, label="OOT"):
    """x vs y at the same approval rate, each with its own cut-off. Net bads avoided by x = bads declined by x - by y.
    Paired bootstrap: the same stratified GOOD/BAD resamples for both models."""
    sx, sy, st = F[f"score_{x}"].to_numpy(), F[f"score_{y}"].to_numpy(), F["sample_status"].to_numpy()
    gb, bad = np.isin(st, LABELLED), st == "BAD"
    bad_gb = bad[gb].astype(int)
    rows, swaps = [], []
    for r in GRID:
        ax, ay = sx >= cuts_x[r], sy >= cuts_y[r]
        dvec = bad_gb * ((~ax[gb]).astype(int) - (~ay[gb]).astype(int))      # +1 = bad declined by x only
        boot = dvec[I].sum(axis=1)
        bx, by = int((bad & ~ax).sum()), int((bad & ~ay).sum())
        rows.append({"split": label, "pair": f"{x} vs {y}", "approval_rate_target": r, f"cutoff_{x}": cuts_x[r], f"cutoff_{y}": cuts_y[r],
                     f"approval_rate_{x}": ax.mean(), f"approval_rate_{y}": ay.mean(),
                     "bads_total": int(bad.sum()), f"bads_declined_{x}": bx, f"bads_declined_{y}": by,
                     f"net_bads_avoided_by_{x}": bx - by, "net_ci_low": float(np.percentile(boot, 2.5)),
                     "net_ci_high": float(np.percentile(boot, 97.5)), "p_delta_le_0": float((boot <= 0).mean()),
                     f"approved_bad_rate_{x}": (bad & ax).sum() / (gb & ax).sum(), f"approved_bad_rate_{y}": (bad & ay).sum() / (gb & ay).sum()})
        for g, mk in {f"declined by {x} only": ~ax & ay, f"declined by {y} only": ax & ~ay,
                      "approved by both": ax & ay, "declined by both": ~ax & ~ay}.items():
            c = _counts(st, mk)
            ngb = c["GOOD"] + c["BAD"]
            swaps.append({"split": label, "pair": f"{x} vs {y}", "approval_rate_target": r, "group": g, "n": int(mk.sum()), **c,
                          "n_good_bad": ngb, "bad_rate": c["BAD"] / ngb if ngb else np.nan})
    return pd.DataFrame(rows), pd.DataFrame(swaps)

def agreement(F, label="OOT", x="A", y="C"):
    """Spearman of the two scores and a decile cross-table (rank deciles, 1 = lowest risk: calibrated PD for the
    models, -score for a benchmark without PD)."""
    def risk(m):
        return F[f"pd_{m}"] if F[f"pd_{m}"].notna().all() else -F[f"score_{m}"]
    dec = {m: pd.qcut(risk(m).rank(method="first"), 10, labels=range(1, 11)).astype(int).to_numpy() for m in (x, y)}
    diff = np.abs(dec[x] - dec[y])
    gb = F["sample_status"].isin(LABELLED).to_numpy()
    summ = {"split": label, "pair": f"{x} vs {y}", "n": len(F),
            "spearman_score_all": float(spearmanr(F[f"score_{x}"], F[f"score_{y}"]).correlation),
            "spearman_score_good_bad": float(spearmanr(F.loc[gb, f"score_{x}"], F.loc[gb, f"score_{y}"]).correlation),
            "decile_diagonal_share": float((diff == 0).mean()), "decile_diagonal_adjacent_share": float((diff <= 1).mean())}
    ct = pd.crosstab(pd.Series(dec[x], name=f"decile_{x} (rows)"), pd.Series(dec[y], name=f"decile_{y}")).reset_index()
    return summ, ct

C_VS_A, SWAP = compare_models(OOT, "C", "A", I_OOT, CUTS["C"], CUTS["A"])
AG_SUMM, AG_CT = agreement(OOT)
for (r, g), t in SWAP.groupby(["approval_rate_target", "split"]):
    assert t["n"].sum() == len(OOT), "swap groups do not reconcile"
# 3.2 Benchmark: A vs A-Score and C vs A-Score on the same OOT applicants (those with an A-Score),
# every score with its own cut-off at the same approval rate on that population
VS_ASCORE, VS_SWAP, VS_AG = None, None, []
if BENCH:
    OOT_B = OOT[OOT[f"has_{BENCH}"]].reset_index(drop=True)
    gbb = OOT_B["sample_status"].isin(LABELLED).to_numpy()
    I_B = boot_idx((OOT_B["sample_status"] == "BAD").to_numpy()[gbb], 2)
    ALL_M = MODELS_USED + [BENCH]
    BENCH_GRID = {m: grid_table(OOT_B, m, I_B, view=f"OOT with {BENCH_NAME} (n={len(OOT_B)})") for m in ALL_M}
    CUTS_B = {m: dict(zip(BENCH_GRID[m]["approval_rate_target"], BENCH_GRID[m]["cutoff_score"])) for m in ALL_M}
    BENCH_BANDS = band_table(OOT_B, BENCH)
    VS_ASCORE, VS_SWAP = {}, []
    for x in MODELS_USED:
        c, s = compare_models(OOT_B, x, BENCH, I_B, CUTS_B[x], CUTS_B[BENCH], label=f"OOT with {BENCH_NAME}")
        VS_ASCORE[x] = c
        VS_SWAP.append(s)
        a, ct = agreement(OOT_B, label=f"OOT with {BENCH_NAME}", x=x, y=BENCH)
        VS_AG.append((a, ct))
    VS_SWAP = pd.concat(VS_SWAP)
    for (p, r), t in VS_SWAP.groupby(["pair", "approval_rate_target"]):
        assert t["n"].sum() == len(OOT_B), "benchmark swap groups do not reconcile"
    for x in MODELS_USED:
        display(VS_ASCORE[x])
print(AG_SUMM)
display(C_VS_A)

In [0]:
QB, QG, QC, QS, QA, QCT = [], [], [], [], [], []
for k, q in enumerate(SPLITS):
    Fq = OOT[OOT["oot_split"] == q].reset_index(drop=True)
    gbq = Fq["sample_status"].isin(LABELLED).to_numpy()
    Iq = boot_idx((Fq["sample_status"] == "BAD").to_numpy()[gbq], 20 + k)
    for m in MODELS_USED:
        QB.append(band_table(Fq, m).assign(split=q, model=m))
        QG.append(grid_table(Fq, m, Iq, cuts=CUTS[m], view=f"OOT {q} (full-OOT cut-offs)"))
    c, s = compare_models(Fq, "C", "A", Iq, CUTS["C"], CUTS["A"], label=q)
    QC.append(c); QS.append(s)
    a, ct = agreement(Fq, label=q)
    QA.append(a); QCT.append((f"Decile cross-table A (rows) vs C, {q}", ct))
QB, QG, QC, QS, QA = pd.concat(QB), pd.concat(QG), pd.concat(QC), pd.concat(QS), pd.DataFrame(QA)

# Benchmark by quarter: same applicants with an A-Score, cut-offs fixed from the full covered OOT population
QVB = []
if BENCH:
    for k, q in enumerate(SPLITS):
        Fq = OOT_B[OOT_B["oot_split"] == q].reset_index(drop=True)
        gbq = Fq["sample_status"].isin(LABELLED).to_numpy()
        Iq = boot_idx((Fq["sample_status"] == "BAD").to_numpy()[gbq], 30 + k)
        QVB.append((f"{BENCH_NAME} grid, {q} (n={len(Fq)})", grid_table(Fq, BENCH, Iq, cuts=CUTS_B[BENCH], view=f"OOT {q} with {BENCH_NAME}")))
        for x in MODELS_USED:
            c, s = compare_models(Fq, x, BENCH, Iq, CUTS_B[x], CUTS_B[BENCH], label=q)
            a, _ = agreement(Fq, label=q, x=x, y=BENCH)
            QVB += [(f"{x} vs {BENCH_NAME}, {q}", c), (f"Swap sets {x} vs {BENCH_NAME}, {q}", s), (f"Agreement {x} vs {BENCH_NAME}, {q}", pd.DataFrame([a]))]
display(QG[["view", "model", "approval_rate_target", "cutoff_score", "approval_rate_achieved", "approved_bad_rate",
            "approved_bad_rate_ci_low", "approved_bad_rate_ci_high", "share_of_bads_declined"]])
display(QC)

In [0]:
imm = (OOT["sample_status"] == "IMMATURE").to_numpy()
IMM_RATE = imm.mean()

def gap_stats(g):
    return pd.Series({"gap_days_mean": g.mean(), "gap_days_median": g.median(), "gap_days_p90": g.quantile(0.9), "gap_days_max": g.max()}) if len(g) else pd.Series(dtype=float)

rows = []
for mon, g in OOT.groupby("app_month"):
    i = g["sample_status"] == "IMMATURE"
    ad = pd.to_datetime(g.loc[i, DECISION_A])
    rows.append({"app_month": mon, "n": len(g), **_counts(g["sample_status"]), "immature_rate": i.mean(),
                 **gap_stats(g.loc[i, "gap_days"]).add_prefix("IMMATURE_").to_dict(),
                 **gap_stats(g.loc[~i, "gap_days"]).add_prefix("others_").to_dict(),
                 "IMMATURE_approvedate_min": ad.min(), "IMMATURE_approvedate_max": ad.max()})
IMM_MONTH = pd.DataFrame(rows)
IMM_APPROVE = (OOT.assign(imm=imm).groupby("approve_month").agg(n=("imm", "size"), IMMATURE=("imm", "sum"))
               .assign(immature_rate=lambda d: d["IMMATURE"] / d["n"]).reset_index())
IMM_GAP = pd.DataFrame([gap_stats(OOT.loc[imm, "gap_days"]).rename("IMMATURE"),
                        gap_stats(OOT.loc[~imm, "gap_days"]).rename("all other OOT")]).reset_index(names="group")

IMM_BAND, FLAGS = [], []
for m in MODELS_USED:
    lb = band_lower(OOT[f"score_{m}"])
    t = pd.DataFrame({"band_lower": lb, "imm": imm}).groupby("band_lower").agg(n=("imm", "size"), IMMATURE=("imm", "sum")).reset_index()
    t = t.sort_values("band_lower", ascending=False)
    t["band"] = t["band_lower"].map(lambda b: f"{b}-{b + BW - 1}")
    t["immature_rate"] = t["IMMATURE"] / t["n"]
    t["ratio_to_overall"] = t["immature_rate"] / IMM_RATE
    t["flag_concentrated"] = (t["n"] >= IMM_BAND_MIN_N) & (t["ratio_to_overall"] >= IMM_RATIO_FLAG)
    IMM_BAND.append(t.assign(model=m))
    tt = t[t["n"] >= IMM_BAND_MIN_N]
    p = chi2_contingency(np.c_[tt["IMMATURE"], tt["n"] - tt["IMMATURE"]])[1]
    FLAGS.append({"check": f"IMMATURE by {m} score band (bands with n >= {IMM_BAND_MIN_N}, chi-square)", "p_value": p,
                  "detail": f"bands with rate >= {IMM_RATIO_FLAG}x overall: {', '.join(t.loc[t['flag_concentrated'], 'band']) or 'none'}",
                  "flag": bool(p < IMM_P_FLAG and t["flag_concentrated"].any())})
IMM_BAND = pd.concat(IMM_BAND)
q = OOT.assign(imm=imm).groupby("oot_split").agg(n=("imm", "size"), IMMATURE=("imm", "sum"))
q["immature_rate"] = q["IMMATURE"] / q["n"]
q["share_of_all_IMMATURE"] = q["IMMATURE"] / imm.sum()
q["share_of_base"] = q["n"] / len(OOT)
p_q = chi2_contingency(np.c_[q["IMMATURE"], q["n"] - q["IMMATURE"]])[1]
r_q = q.loc["Q2", "immature_rate"] / q.loc["Q1", "immature_rate"] if q.loc["Q1", "immature_rate"] > 0 else np.inf
FLAGS.insert(0, {"check": "IMMATURE concentrated in Q2 (rate Q2 / Q1, chi-square)", "p_value": p_q,
                 "detail": f"rate Q1 {q.loc['Q1', 'immature_rate']:.4f}, Q2 {q.loc['Q2', 'immature_rate']:.4f}, ratio {r_q:.2f}; "
                           f"Q2 holds {q.loc['Q2', 'share_of_all_IMMATURE']:.1%} of IMMATURE vs {q.loc['Q2', 'share_of_base']:.1%} of the base",
                 "flag": bool(p_q < IMM_P_FLAG and r_q >= IMM_RATIO_FLAG)})
IMM_Q = q.reset_index()
IMM_FLAGS = pd.DataFrame(FLAGS)
display(IMM_FLAGS)
display(IMM_MONTH[["app_month", "n", "IMMATURE", "immature_rate", "IMMATURE_gap_days_median", "others_gap_days_median", "IMMATURE_approvedate_min", "IMMATURE_approvedate_max"]])

In [0]:
# Diagnosis only: no labels are used for FWD / STRESS. PSI on the BW-point score bands, OOT base as reference.
# Benchmark: only applicants with an A-Score, cut-offs from the covered OOT population (CUTS_B).
REC = []
for m in MODELS_USED + ([BENCH] if BENCH else []):
    sel = (lambda df: df[df[f"has_{BENCH}"]]) if m == BENCH else (lambda df: df)
    cuts_m = CUTS_B[BENCH] if m == BENCH else CUTS[m]
    O, Fw, S = sel(OOT), sel(FWD), sel(STRESS)
    ref = pd.Series(band_lower(O[f"score_{m}"]))
    groups = [("OOT", "ALL", O)] + [(w, "ALL", df) for w, df in (("FWD", Fw), ("STRESS", S)) if len(df)] + \
             [(w, mon, g) for w, df in (("FWD", Fw), ("STRESS", S)) for mon, g in df.groupby("app_month")]
    for w, mon, g in groups:
        s = g[f"score_{m}"].to_numpy()
        row = {"model": BENCH_NAME if m == BENCH else m, "window": w, "app_month": mon, "n": len(g), **_counts(g["sample_status"]),
               "mean_score": s.mean(), "median_score": float(np.median(s)),
               "score_psi_vs_oot": common.psi_from_bins(ref, pd.Series(band_lower(s)), EPS)}
        for r in GRID:                                   # cut-off of each model at rate r: see grid_oot_<model>
            row[f"share_ge_cutoff_at_{r:g}"] = float((s >= cuts_m[r]).mean())
        REC.append(row)
REC = pd.DataFrame(REC)
display(REC[REC["app_month"] == "ALL"])

In [0]:
def watch_frame(df):
    out = {}
    for w in WATCH_COLS:
        if w.upper() == "AGE":
            b = pd.cut(pd.to_numeric(df[w], errors="coerce"), AGE_BANDS, right=False)
            out["AGE_BAND"] = b.astype(object).where(b.notna(), "MISSING").astype(str)
        else:
            v = _norm(df[w])
            v = v.astype(object).where(v.notna(), "MISSING").astype(str)
            top = v.value_counts().index[:TOP_N_WATCH]
            out[w] = v.where(v.isin(top), "OTHER")
    return pd.DataFrame(out, index=df.index)

WF = watch_frame(OOT)
st_all = OOT["sample_status"].to_numpy()
SEGROWS = []
key = 100
for dim in WF.columns:
    for val in sorted(WF[dim].unique()):
        for m in MODELS_USED + ([BENCH] if BENCH else []):
            # benchmark rows: applicants with an A-Score only, cut-offs from the covered OOT population
            cov = OOT[f"has_{BENCH}"].to_numpy() if m == BENCH else np.ones(len(OOT), bool)
            cuts_m = CUTS_B[BENCH] if m == BENCH else CUTS[m]
            mk = (WF[dim] == val).to_numpy() & cov
            if not mk.any():
                continue
            gbm, badm = np.isin(st_all, LABELLED) & mk, (st_all == "BAD") & mk
            n_bads = int(badm.sum())
            with_ci = n_bads >= SEG_MIN_BADS
            key += 1
            I_s = boot_idx(badm[gbm], key) if with_ci else None
            sc = OOT[f"score_{m}"].to_numpy()
            for r in GRID:
                a = sc >= cuts_m[r]
                gb_app, b_app = int((a & gbm).sum()), int((a & badm).sum())
                row = {"dim": dim, "value": val, "model": BENCH_NAME if m == BENCH else m, "approval_rate_target": r, "cutoff_score": cuts_m[r],
                       "n": int(mk.sum()), "n_approved": int((a & mk).sum()), "n_declined": int((~a & mk).sum()),
                       "approval_rate": (a & mk).sum() / mk.sum(), "approval_rate_good_bad_only": gb_app / gbm.sum() if gbm.any() else np.nan,
                       "good_bad": int(gbm.sum()), "bads": n_bads, "bads_approved": b_app, "bads_declined": n_bads - b_app,
                       "approved_bad_rate": b_app / gb_app if gb_app else np.nan,
                       "note": "" if with_ci else f"< {SEG_MIN_BADS} bads: bad rate and count only"}
                if with_ci:
                    ag = a[gbm]
                    br = (badm[gbm] & ag)[I_s].sum(axis=1) / np.where(ag[I_s].sum(axis=1) == 0, np.nan, ag[I_s].sum(axis=1))
                    row["approved_bad_rate_ci_low"], row["approved_bad_rate_ci_high"] = np.nanpercentile(br, 2.5), np.nanpercentile(br, 97.5)
                else:
                    row["approved_bad_rate_ci_low"] = row["approved_bad_rate_ci_high"] = np.nan
                SEGROWS.append(row)
SEGROWS = pd.DataFrame(SEGROWS)
SEG_AVAIL = pd.DataFrame([{"segment_column": c, "in_acard_master": c in m_cols} for c in SEGMENTS])
display(SEGROWS[(SEGROWS["approval_rate_target"] == 0.8) & (SEGROWS["model"] == "A")])

In [0]:
def pct_ci(v, lo, hi):
    return f"{v:.2%} [{lo:.2%}, {hi:.2%}]"

SUMMARY = []
for r in GRID:
    row = {"approval_rate_target": r}
    for m in MODELS_USED:
        g = GRID_OOT[m].set_index("approval_rate_target").loc[r]
        row.update({f"{m}_cutoff_score": int(g["cutoff_score"]), f"{m}_approval_rate_achieved": g["approval_rate_achieved"],
                    f"{m}_approved_bad_rate": g["approved_bad_rate"],
                    f"{m}_approved_bad_rate_CI": pct_ci(g["approved_bad_rate"], g["approved_bad_rate_ci_low"], g["approved_bad_rate_ci_high"]),
                    f"{m}_share_of_bads_declined": g["share_of_bads_declined"],
                    f"{m}_INDETERMINATE_share": g["INDETERMINATE_share_among_approved"],
                    f"{m}_IMMATURE_share": g["IMMATURE_share_among_approved"],
                    f"{m}_EXCLUDED_PERF_share": g["EXCLUDED_PERF_share_among_approved"]})
    c = C_VS_A.set_index("approval_rate_target").loc[r]
    row.update({"net_bads_avoided_C_vs_A": int(c["net_bads_avoided_by_C"]),
                "net_bads_avoided_C_vs_A_CI": f"{int(c['net_bads_avoided_by_C'])} [{c['net_ci_low']:g}, {c['net_ci_high']:g}]",
                "P(delta <= 0)": c["p_delta_le_0"]})
    if BENCH:      # on the OOT applicants with an A-Score; every score with its own cut-off on that population
        g = BENCH_GRID[BENCH].set_index("approval_rate_target").loc[r]
        row.update({f"{BENCH_NAME}_cutoff_score": int(g["cutoff_score"]), f"{BENCH_NAME}_approval_rate_achieved": g["approval_rate_achieved"],
                    f"{BENCH_NAME}_approved_bad_rate_CI": pct_ci(g["approved_bad_rate"], g["approved_bad_rate_ci_low"], g["approved_bad_rate_ci_high"]),
                    f"{BENCH_NAME}_share_of_bads_declined": g["share_of_bads_declined"]})
        for x in MODELS_USED:
            v = VS_ASCORE[x].set_index("approval_rate_target").loc[r]
            row.update({f"net_bads_avoided_{x}_vs_{BENCH_NAME}_CI": f"{int(v[f'net_bads_avoided_by_{x}'])} [{v['net_ci_low']:g}, {v['net_ci_high']:g}]",
                        f"P(delta <= 0) {x} vs {BENCH_NAME}": v["p_delta_le_0"]})
        row[f"{BENCH_NAME} comparison population"] = f"OOT with {BENCH_NAME}: {len(OOT_B)} of {len(OOT)}"
    else:
        row["vs_A_Score"] = "not available: config.cutoff_analysis.compare_with is null"
    SUMMARY.append(row)
SUMMARY = pd.DataFrame(SUMMARY)

META = [("module", "s06b_cutoff_analysis (spec 8.2, 8.3, 9.1-9.3; E13 scoring code). REPORT ONLY."),
        ("caveat", CAVEAT), ("run_time", RUN_TIME), ("git_commit", GIT_COMMIT), ("config_sha256", CONFIG_HASH),
        ("feature_dict_sha256", FDICT_SHA), ("registry", REG_PATH), ("registry version / locked_at", f"{REG['version']} / {REG['locked_at']}"),
        ("registry sha256", REG_SHA), ("sign-off", f"{REG['signoff']['decision_rules_signed_by']} {REG['signoff']['signed_date']}"),
        ("locked files hash-verified", len(FILES)), ("Platt file / sha256", f"{PLATT_PATH} / {PLATT_SHA}"),
        ("models", ", ".join(f"{m}: Platt a={PLATT[m]['a']:.4f}, b={PLATT[m]['b']:.4f}" for m in MODELS_USED)),
        ("score scaling", json.dumps(cfg["scaling"])),
        ("input", f"{MASTER_TBL} (Delta v{PROTECTED_TABLES.get(MASTER_TBL)}), approved applicants only, via common.load_window"),
        ("approval-rate base", f"sample_status in {BASE_STATUSES} (= config.psi_statuses); EXCLUDED not used (owner 2026-11-17)"),
        ("bad rates", "GOOD + BAD only (config.labelled_statuses); INDETERMINATE, IMMATURE, EXCLUDED_PERF reported as separate counts"),
        ("OOT window", f"{cfg['windows']['OOT']} on {DATE_A}; base {len(OOT)} = {OOT_COUNTS.to_dict()}; splits {SPLITS}"),
        ("OOT vs spec 1.2 confirmed counts", f"GOOD+BAD {int(OOT_COUNTS['GOOD'] + OOT_COUNTS['BAD'])} / bads {int(OOT_COUNTS['BAD'])} vs config.expected_counts "
                                              f"{cfg.get('expected_counts', {}).get('OOT')} (same rows as E13; known E2 difference)"),
        ("DEV second view", f"oof_predictions.parquet p_oof (models A, C) + Platt; base = DEV GOOD+BAD ({len(DEVO['A'])}); other statuses have no OOF prediction; no in-sample DEV scores"),
        ("FWD / STRESS", f"score distribution only (no labels): FWD {len(FWD)}, STRESS {len(STRESS)}, same base statuses"),
        ("cut-off rule", "approve if score >= cut-off; cut-off = highest integer score with approval rate >= target; ties reported"),
        ("bootstrap", f"{BOOT}, stratified on GOOD/BAD, cut-offs fixed; C vs A paired (same resamples)"),
        ("by_quarter", "full-OOT cut-offs applied to each quarter"),
        ("agreement deciles", "rank deciles of calibrated PD on the OOT base (1 = lowest PD)"),
        ("segments", f"WATCH only {WATCH_COLS}; config.columns.segments {SEG_MISSING} not in acard_master (open question)"),
        ("compare_with", str(COMPARE_WITH)),
        ("benchmark", (f"{BENCH_NAME} from {COMPARE_WITH['table']}.{COMPARE_WITH['score_column']} joined on {COMPARE_WITH['key']} "
                       f"(owner's load_a_score); used only if {COMPARE_WITH['date_column']} month - application month in {BENCH_OFFSETS} "
                       f"(owner 2026-11-17); compared on the {len(OOT_B)} OOT applicants with a usable score ({len(OOT_B) / len(OOT):.1%}); "
                       f"no PD (expected bad rate n/a); no DEV view (not an OOF score)") if BENCH else "none"),
        ("report-only constants", f"SEG_MIN_BADS {SEG_MIN_BADS}; AGE_BANDS {AGE_BANDS}; TOP_N_WATCH {TOP_N_WATCH}; IMMATURE flags ratio {IMM_RATIO_FLAG}, min n {IMM_BAND_MIN_N}, p {IMM_P_FLAG}")]
META = pd.DataFrame(META, columns=["key", "value"])

SHEETS = {"meta": [("Run metadata", META), ("Status counts in acard_master by window", STATUS_COUNTS), ("E13 spot check", SPOT)],
          "summary": SUMMARY}
for m in MODELS_USED:
    SHEETS[f"score_bands_{m}"] = BANDS[m]
for m in MODELS_USED:
    SHEETS[f"grid_oot_{m}"] = GRID_OOT[m]
for m in MODELS_USED:
    SHEETS[f"grid_dev_oof_{m}"] = GRID_DEV[m]
SHEETS["C_vs_A"] = C_VS_A
if BENCH:
    SHEETS[f"score_bands_{BENCH_NAME}"[:31]] = BENCH_BANDS
    SHEETS["vs_a_score"] = ([(f"Coverage of {BENCH_NAME}", BENCH_COVERAGE), ("Score month minus application month", BENCH_TIMING)]
                            + [(f"Grid on OOT with {BENCH_NAME}: {m if m != BENCH else BENCH_NAME}", BENCH_GRID[m]) for m in ALL_M]
                            + [(f"{x} vs {BENCH_NAME}", VS_ASCORE[x]) for x in MODELS_USED]
                            + [(f"Swap sets vs {BENCH_NAME}", VS_SWAP)]
                            + [(f"Agreement {a['pair']}", pd.DataFrame([a])) for a, _ in VS_AG]
                            + [(f"Decile cross-table {a['pair']}", ct) for a, ct in VS_AG]
                            + [(f"By quarter: {t}", d) for t, d in QVB])
SHEETS.update({
    "swap_sets": SWAP,
    "agreement": [("Score agreement, OOT", pd.DataFrame([AG_SUMM])), ("Decile cross-table A (rows) vs C, OOT", AG_CT)],
    "by_quarter": [("Score bands by quarter", QB), ("Approval-rate grid by quarter (full-OOT cut-offs)", QG),
                   ("C vs A by quarter", QC), ("Swap sets by quarter", QS), ("Agreement by quarter", QA)] + QCT,
    "oot_immature": [("Flags", IMM_FLAGS), ("By OOT quarter", IMM_Q), ("By application month (gap = APPROVEDATE - APPRECDDATE, days)", IMM_MONTH),
                     ("Gap application -> approval, IMMATURE vs other OOT", IMM_GAP), ("By approval month", IMM_APPROVE),
                     ("By score band", IMM_BAND)],
    "recent_distribution": [("Cut-offs per model (from grid_oot)", pd.DataFrame(CUTS).reset_index(names="approval_rate_target")),
                            ("Score distribution vs OOT (diagnosis only, no labels)", REC)],
    "segments": [("Segment columns (config.columns.segments)", SEG_AVAIL), ("By WATCH variable at each cut-off", SEGROWS)]})
save_excel(SHEETS, XLSX_PATH)

# Open question requested by the owner: append once, replace the line on re-runs (idempotent)
TAG = "[S06b-segments]"
line = (f"- {TAG} s06b_cutoff_analysis ({RUN_TIME[:10]}): config.columns.segments {SEG_MISSING} are not columns of {MASTER_TBL}. "
        f"The cut-off segment analysis uses the WATCH variables only ({', '.join(WATCH_COLS)}). Should these segment columns be added "
        f"to acard_master (and from which source), or removed from config.columns.segments?")
old = open(OQ_PATH).read() if os.path.exists(OQ_PATH) else "# Open questions\n"
lines = [l for l in old.rstrip("\n").split("\n") if TAG not in l]
if SEG_MISSING:
    lines += ["", line] if lines and lines[-1].strip() else [line]
write_text("\n".join(lines) + "\n", OQ_PATH)

import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="s06b_cutoff_analysis") as run:
    mlflow.log_params({"module": "s06b_cutoff_analysis", "registry_version": REG["version"], "models": ",".join(MODELS_USED),
                       "approval_rate_grid": ",".join(map(str, GRID)), "score_band_width": BW, "base_statuses": ",".join(BASE_STATUSES),
                       "config_sha256": CONFIG_HASH, "git_commit": GIT_COMMIT, "registry_sha256": REG_SHA})
    for m in MODELS_USED:
        for _, g in GRID_OOT[m].iterrows():
            k = f"{int(round(g['approval_rate_target'] * 1000))}"
            mlflow.log_metrics({f"{m}_cutoff_{k}": g["cutoff_score"], f"{m}_approved_bad_rate_{k}": g["approved_bad_rate"],
                                f"{m}_share_bads_declined_{k}": g["share_of_bads_declined"]})
    for _, c in C_VS_A.iterrows():
        mlflow.log_metric(f"net_bads_avoided_C_vs_A_{int(round(c['approval_rate_target'] * 1000))}", c["net_bads_avoided_by_C"])
    if BENCH:
        for x in MODELS_USED:
            for _, v in VS_ASCORE[x].iterrows():
                mlflow.log_metric(f"net_bads_avoided_{x}_vs_ascore_{int(round(v['approval_rate_target'] * 1000))}", v[f"net_bads_avoided_by_{x}"])
    mlflow.log_artifact(XLSX_PATH)
    MLFLOW_RUN_ID = run.info.run_id
print(f"Workbook: {XLSX_PATH} ({len(SHEETS)} sheets) | MLflow run {MLFLOW_RUN_ID}")
display(SUMMARY)

In [0]:
import openpyxl
CHECKS = {}
# 1. Registry, locked files, calibration unchanged; nothing written outside the output folder
CHECKS["registry file unchanged"] = sha256_file(REG_PATH) == REG_SHA
CHECKS["all locked model/binning/fold/OOF file hashes match the registry"] = all(
    sha256_file(v["path"]) == v["sha256"] for v in read_json(REG_PATH)["files"].values())
CHECKS["Platt calibration file unchanged (registry hash)"] = sha256_file(PLATT_PATH) == PLATT_SHA == CAL_REF["sha256"]
CHECKS["no file created/changed under the model folder outside the output folder"] = tree_snapshot(MODEL_ROOT) == SNAP0
CHECKS["protected Delta tables unchanged (acard_master, feature logs, E13 predictions)"] = {t: table_version(t) for t in PROTECTED_TABLES} == PROTECTED_TABLES
CHECKS["every file written is inside the output folder (except the requested open_questions.md entry)"] = all(
    os.path.abspath(p).startswith(os.path.abspath(OUT) + os.sep) or os.path.abspath(p) == os.path.abspath(OQ_PATH) for p in WRITTEN) and os.path.exists(XLSX_PATH)

# 2. OOT scores reproduce the stored E13 scores (1,000 rows)
for _, rw in SPOT.iterrows():
    CHECKS[f"{rw['model']}: E13 spot check (1,000 rows, exact score, PD within 1e-9)"] = bool(
        rw["n"] == 1000 and rw["n_score_diff"] == 0 and rw["max_abs_pd_diff"] <= 1e-9)

# 3. Reconciliation for every model and cut-off
G_ALL = pd.concat([*GRID_OOT.values(), *GRID_DEV.values(), QG] + ([*BENCH_GRID.values()] + [d for t, d in QVB if "grid" in t] if BENCH else []), ignore_index=True)
CHECKS["grids: approved + declined = total (every model, cut-off, view)"] = bool((G_ALL["n_approved"] + G_ALL["n_declined"] == G_ALL["n_total"]).all())
CHECKS["grids: bads approved + bads declined = total bads"] = bool((G_ALL["bads_approved"] + G_ALL["bads_declined"] == G_ALL["bads_total"]).all())
CHECKS["grids: GOOD+BAD approved + declined = total GOOD+BAD"] = bool((G_ALL["good_bad_approved"] + G_ALL["good_bad_declined"] == G_ALL["good_bad_total"]).all())
CHECKS["OOT grids: total = OOT base and total bads = OOT BAD"] = all(
    (g["n_total"] == len(OOT)).all() and (g["bads_total"] == OOT_COUNTS["BAD"]).all() for g in GRID_OOT.values())
CHECKS["OOT grids: other statuses approved + declined = their OOT counts"] = all(
    (g[f"{s}_approved"] + g[f"{s}_declined"] == OOT_COUNTS[s]).all() for g in GRID_OOT.values() for s in OTHER)
CHECKS["score bands reconcile to the OOT base and bads"] = all(
    t["n"].sum() == len(OOT) and t["BAD"].sum() == OOT_COUNTS["BAD"] and t["cum_bads_declined"].iloc[-1] == 0 for t in BANDS.values())
sw = pd.concat([SWAP, QS]).groupby(["split", "approval_rate_target"])["n"].sum()
split_n = {"OOT": len(OOT), **OOT["oot_split"].value_counts().to_dict()}
CHECKS["swap sets: four groups reconcile to the population (OOT and each quarter)"] = all(v == split_n[s] for (s, _), v in sw.items())
CHECKS["segments: approved + declined = segment n; bads approved + declined = segment bads"] = bool(
    ((SEGROWS["n_approved"] + SEGROWS["n_declined"]) == SEGROWS["n"]).all() and ((SEGROWS["bads_approved"] + SEGROWS["bads_declined"]) == SEGROWS["bads"]).all())
_segn = SEGROWS.groupby(["dim", "model", "approval_rate_target"])["n"].sum().reset_index()
_segn["expected"] = np.where(_segn["model"] == BENCH_NAME, len(OOT_B) if BENCH else -1, len(OOT))
CHECKS["segments: each WATCH dimension covers the OOT base (benchmark: OOT with A-Score)"] = bool((_segn["n"] == _segn["expected"]).all())
if BENCH:
    CHECKS["benchmark: comparison population = OOT rows with an A-Score, same rows for every score"] = bool(
        len(OOT_B) == int(OOT[f"has_{BENCH}"].sum()) and all((g["n_total"] == len(OOT_B)).all() for g in BENCH_GRID.values()))
    CHECKS["benchmark: every A-Score used has score month - application month in the allowed offsets"] = bool(
        OOT_B[f"bench_month_offset_{BENCH}"].isin(BENCH_OFFSETS).all()
        and all(df.loc[df[f"has_{BENCH}"], f"bench_month_offset_{BENCH}"].isin(BENCH_OFFSETS).all() for df in (FWD, STRESS)))
    CHECKS["benchmark: swap sets reconcile to the comparison population"] = bool(
        (VS_SWAP.groupby(["pair", "approval_rate_target"])["n"].sum() == len(OOT_B)).all())
    CHECKS["benchmark: A-Score not used to score the models (model scores unchanged by the merge)"] = all(
        (OOT[f"score_{m}"].to_numpy() == common.pd_to_score(OOT[f"pd_{m}"].to_numpy(), cfg)).all() for m in MODELS_USED)

# 4. Bad rates use OOT GOOD/BAD or DEV OOF only; no in-sample DEV scores
CHECKS["no DEV rows scored with the models (DEV only via OOF predictions)"] = "DEV" not in SCORED_WINDOWS and set(OOT["window"]) == {"OOT"}
CHECKS["DEV view PD = Platt(p_oof) for every row"] = all(
    np.allclose(DEVO[m][f"pd_{m}"], common.apply_platt(DEVO[m]["p_oof"].to_numpy(), PLATT[m]), atol=0, rtol=1e-12) for m in MODELS_USED)
_chk = []                                                        # independent recomputation with pandas filters
for m in MODELS_USED:
    for r in (GRID[0], GRID[-1]):
        c = CUTS[m][r]
        lab = OOT[OOT["sample_status"].isin(["GOOD", "BAD"]) & (OOT[f"score_{m}"] >= c)]
        _chk.append(np.isclose((lab["sample_status"] == "BAD").mean(),
                               GRID_OOT[m].set_index("approval_rate_target").loc[r, "approved_bad_rate"]))
CHECKS["approved bad rate = BAD / (GOOD + BAD) among approved (independent recomputation)"] = all(_chk)

# 5. Workbook: required sheets and the approved-only caveat on every sheet
wb = openpyxl.load_workbook(XLSX_PATH, read_only=True)
req = ["meta", "summary", "score_bands_A", "score_bands_C", "grid_oot_A", "grid_oot_C", "grid_dev_oof_A", "grid_dev_oof_C", "C_vs_A",
       "swap_sets", "agreement", "by_quarter", "oot_immature", "recent_distribution", "segments"] + (["vs_a_score"] if COMPARE_WITH else [])
CHECKS["workbook has every required sheet"] = all(s in wb.sheetnames for s in req)
CHECKS["approved-only caveat in cell A1 of every sheet"] = all(next(wb[s].iter_rows(min_row=1, max_row=1, values_only=True))[0] == CAVEAT for s in wb.sheetnames)
wb.close()

RES = pd.DataFrame([{"check": k, "pass": bool(v)} for k, v in CHECKS.items()])
display(RES)
ALL_PASS = bool(RES["pass"].all())
write_json({"module": "s06b_cutoff_analysis", "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
            "registry": {"path": REG_PATH, "version": REG["version"], "sha256": REG_SHA},
            "rows": {"oot_base": len(OOT), "oot_by_status": OOT_COUNTS.to_dict(), "fwd": len(FWD), "stress": len(STRESS),
                     "dev_oof": {m: len(d) for m, d in DEVO.items()}},
            "cutoffs": {m: {str(k): int(v) for k, v in CUTS[m].items()} for m in MODELS_USED},
            "approved_bad_rate": {m: dict(zip(map(str, GRID_OOT[m]["approval_rate_target"]), GRID_OOT[m]["approved_bad_rate"])) for m in MODELS_USED},
            "net_bads_avoided_C_vs_A": dict(zip(map(str, C_VS_A["approval_rate_target"]), C_VS_A["net_bads_avoided_by_C"])),
            "outputs": WRITTEN + [SUMMARY_PATH], "mlflow_run_id": MLFLOW_RUN_ID,
            "checks": CHECKS, "all_pass": ALL_PASS, "runtime_min": round((time.time() - _t0) / 60, 2)}, SUMMARY_PATH)
failed = RES.loc[~RES["pass"], "check"].tolist()
assert ALL_PASS, f"Acceptance FAILED: {failed}"
print(f"All {len(RES)} acceptance checks PASS | run summary {SUMMARY_PATH}")